# Real2Sim Studio — chạy pipeline trên Kaggle

Xem `scripts/kaggle/README.md`. Bật **GPU** và **Internet**; thêm secret `HF_TOKEN`, `GITHUB_TOKEN`. Sửa các chỗ `<...>` trước khi chạy.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["GITHUB_TOKEN"] = secrets.get_secret("GITHUB_TOKEN")

In [ ]:
%%bash
# Ô 1 — mã nguồn + cài đặt (khoảng 15–25 phút lần đầu; gsplat biên dịch kernel CUDA)
git clone --branch feat/pipeline-v2 https://$GITHUB_TOKEN@github.com/<org>/<repo>.git /kaggle/working/app
cd /kaggle/working/app && bash scripts/kaggle/setup_kaggle.sh /kaggle/working/r2s

In [ ]:
%%bash
# Ô 2 — kiểm máy (phải "đủ điều kiện chạy" mới đi tiếp)
cd /kaggle/working/app && set -a && source /kaggle/working/r2s/kaggle.env && set +a
.venv/bin/python -m scripts.kaggle.preflight

In [ ]:
%%bash
# Ô 3 — lượt 1: video → 3DGS, dừng ở bước chờ xác nhận hệ mét, đẩy gói lên Hugging Face
cd /kaggle/working/app && set -a && source /kaggle/working/r2s/kaggle.env && set +a
.venv/bin/python -m scripts.kaggle.run_job --video /kaggle/input/<dataset>/video.mp4 --name "Kho A" \
  --attestation /kaggle/input/<dataset>/attestation.json \
  --out /kaggle/working/bundles --hf-repo <user>/real2sim-artifacts

In [ ]:
%%bash
# Ô 4 — lượt 2 trên Kaggle (phiên mới thì chạy lại ô 1–2 trước)
cd /kaggle/working/app && set -a && source /kaggle/working/r2s/kaggle.env && set +a
.venv/bin/python -m scripts.kaggle.job_bundle pull <job_id> --repo <user>/real2sim-artifacts --out /kaggle/working/in
.venv/bin/python -m scripts.kaggle.job_bundle import /kaggle/working/in/<job_id>
.venv/bin/python -m scripts.kaggle.run_job --job-id <job_id> --metric-input /kaggle/input/<dataset>/metric.json \
  --out /kaggle/working/bundles --hf-repo <user>/real2sim-artifacts